# Toxoplasma omics catalogue


# Toxoplasma omics studies: what was perturbed, what was measured

Instruction 69. The request (2026-10-17): compare expression between Toxoplasma strains and under perturbations such as a thioredoxin knockout; as many gene-knockout and drug studies with RNA-seq as possible; and other omics readouts. This notebook lists every candidate study from the repositories' own APIs -- no accession typed by hand -- and classifies each, so loading can be prioritised by what is actually there.

Sources: NCBI GEO via E-utilities (sequencing and arrays), PRIDE (proteomics), Metabolomics Workbench (metabolomics). Classification is by keyword on title and summary, and is a first pass for a person to check: the table keeps the text it matched on.

In [1]:
import json, re, time, os, urllib.parse, urllib.request
import pandas as pd
OUT = '/media/carruthers/mnt3/claude/repo/starplast/results/omics_catalogue_2026_10_17'
os.makedirs(OUT, exist_ok=True)
def get(url, tries=4):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return r.read().decode()
        except Exception as e:
            if i == tries - 1: raise
            time.sleep(2 * (i + 1))
QUERIES = []  # every query sent, kept for the record
def logged(url):
    QUERIES.append(url); return get(url)

## 1. GEO series

Two searches, unioned: the organism field (series whose samples are Toxoplasma), and all fields (also series indexed under the host only, where the parasite is the perturbation).

In [2]:
E = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
terms = ['"Toxoplasma gondii"[Organism] AND gse[ETYP]', 'Toxoplasma[All Fields] AND gse[ETYP]']
ids = {}
for t in terms:
    q = f'{E}/esearch.fcgi?' + urllib.parse.urlencode({'db': 'gds', 'term': t, 'retmax': 5000, 'retmode': 'json'})
    r = json.loads(logged(q))['esearchresult']
    ids[t] = set(r['idlist']); time.sleep(0.4)
    print(f"{r['count']:>5} {t}")
uids = sorted(set().union(*ids.values()))
len(uids)

  188 "Toxoplasma gondii"[Organism] AND gse[ETYP]
  371 Toxoplasma[All Fields] AND gse[ETYP]


371

In [3]:
docs = []
for i in range(0, len(uids), 200):
    q = f'{E}/esummary.fcgi?' + urllib.parse.urlencode({'db': 'gds', 'id': ','.join(uids[i:i + 200]), 'retmode': 'json'})
    r = json.loads(logged(q))['result']
    docs += [r[u] for u in r['uids']]; time.sleep(0.4)
geo = pd.DataFrame([{
    'source': 'GEO', 'accession': d.get('accession'), 'title': d.get('title', ''),
    'summary': d.get('summary', ''), 'type': d.get('gdstype', ''), 'samples': int(d.get('n_samples') or 0),
    'taxa': d.get('taxon', ''), 'pubmed': ';'.join(map(str, d.get('pubmedids', []))),
    'supplementary': d.get('suppfile', ''), 'date': d.get('pdat', ''),
    'parasite_measured': 'Toxoplasma' in d.get('taxon', ''),
} for d in docs])
geo['parasite_measured'].value_counts()

parasite_measured
True     190
False    181

## 1b. SRA-only RNA-seq (BioProject)

Some studies deposit reads in SRA without a GEO series -- the Trx1 thioredoxin mutant is one (PRJNA1008424). Transcriptome projects on the organism, minus those already linked to a GEO series.

In [4]:
Q = chr(34)
# Two searches, unioned: the organism field, and all fields -- the Trx1 thioredoxin mutant project
# (PRJNA1008424) has an EMPTY organism field and is found only by the second.
terms_bp = [f'{Q}Toxoplasma gondii{Q}[Organism] AND {Q}transcriptome or gene expression{Q}[Project Data Type]',
            f'Toxoplasma[All Fields] AND ({Q}transcriptome or gene expression{Q}[Project Data Type] OR {Q}raw sequence reads{Q}[Project Data Type]) AND (RNA-seq[All Fields] OR transcriptome[All Fields] OR RNA sequencing[All Fields])']
bp_ids = set()
for t in terms_bp:
    q = f'{E}/esearch.fcgi?' + urllib.parse.urlencode({'db': 'bioproject', 'term': t, 'retmax': 5000, 'retmode': 'json'})
    r = json.loads(logged(q))['esearchresult']; bp_ids |= set(r['idlist']); time.sleep(0.4)
    print(f"{r['count']:>5} {t}")
bp_ids = sorted(bp_ids)
bp = []
for i in range(0, len(bp_ids), 200):
    q = f'{E}/esummary.fcgi?' + urllib.parse.urlencode({'db': 'bioproject', 'id': ','.join(bp_ids[i:i + 200]), 'retmode': 'json'})
    r = json.loads(logged(q))['result']; bp += [r[u] for u in r['uids']]; time.sleep(0.4)
# A GEO series names its BioProject in the summary record's text only sometimes; match on the
# GSE accession through elink instead, which is exact.
linked = set()
for i in range(0, len(bp_ids), 100):
    # One id= per project, so each comes back in its own linkset; a comma-joined id list returns
    # ONE linkset for all of them, which once marked all 125 projects as linked.
    q = f'{E}/elink.fcgi?' + urllib.parse.urlencode([('dbfrom', 'bioproject'), ('db', 'gds'), ('retmode', 'json')] + [('id', x) for x in bp_ids[i:i + 100]])
    for ls in json.loads(logged(q)).get('linksets', []):
        if any(db.get('links') for db in ls.get('linksetdbs', [])): linked.update(map(str, ls.get('ids', [])))
    time.sleep(0.4)
sra_only = pd.DataFrame([{
    'source': 'SRA', 'accession': d.get('project_acc'), 'title': d.get('project_title', ''),
    'summary': d.get('project_description', ''), 'type': 'expression profiling by high throughput sequencing (SRA only)',
    'samples': None, 'taxa': d.get('organism_name', '') or 'Toxoplasma gondii', 'pubmed': '',
    'supplementary': 'raw reads only', 'date': d.get('registration_date', ''), 'parasite_measured': True,
} for d in bp if str(d.get('uid')) not in linked])
len(bp), len(linked), len(sra_only)

  125 "Toxoplasma gondii"[Organism] AND "transcriptome or gene expression"[Project Data Type]
  359 Toxoplasma[All Fields] AND ("transcriptome or gene expression"[Project Data Type] OR "raw sequence reads"[Project Data Type]) AND (RNA-seq[All Fields] OR transcriptome[All Fields] OR RNA sequencing[All Fields])


(359, 254, 105)

## 2. PRIDE (proteomics)

In [5]:
rows, page = [], 0
while True:
    q = 'https://www.ebi.ac.uk/pride/ws/archive/v3/search/projects?' + urllib.parse.urlencode({'keyword': 'Toxoplasma', 'pageSize': 100, 'page': page})
    batch = json.loads(logged(q))
    if not batch: break
    rows += batch; page += 1; time.sleep(0.3)
    if len(batch) < 100: break
pride = pd.DataFrame([{
    'source': 'PRIDE', 'accession': p.get('accession'), 'title': p.get('title', ''),
    'summary': p.get('projectDescription') or '',  # not the protocol: 'treated with trypsin' is not a drug
    'protocol': p.get('sampleProcessingProtocol') or '',
    'type': 'proteomics: ' + ', '.join(p.get('experimentTypes') or []),
    'samples': None, 'taxa': ', '.join(p.get('organisms') or []),
    'pubmed': '', 'supplementary': '', 'date': p.get('submissionDate', ''),
    'parasite_measured': any('toxoplasma' in str(o).lower() for o in (p.get('organisms') or [])),
} for p in rows])
len(pride), int(pride['parasite_measured'].sum())

(209, 182)

## 3. Metabolomics Workbench

In [6]:
mw = []
for field in ('study_title', 'species'):
    q = f'https://www.metabolomicsworkbench.org/rest/study/{field}/Toxoplasma/summary'
    try:
        r = json.loads(logged(q))
    except Exception as e:
        print(field, 'failed:', e); continue
    r = [r] if isinstance(r, dict) and 'study_id' in r else list(r.values()) if isinstance(r, dict) else r
    mw += r; time.sleep(0.3)
meta = pd.DataFrame([{
    'source': 'Metabolomics Workbench', 'accession': s.get('study_id'), 'title': s.get('study_title', ''),
    'summary': s.get('study_summary', ''), 'type': 'metabolomics: ' + str(s.get('analysis_type', '')),
    'samples': int(s['number_of_samples']) if str(s.get('number_of_samples', '')).isdigit() else None,
    'taxa': s.get('species', ''), 'pubmed': '', 'supplementary': '', 'date': s.get('release_date', ''),
    'parasite_measured': 'toxoplasma' in str(s.get('species', '')).lower(),
} for s in mw]).drop_duplicates('accession')
len(meta)

species failed: Expecting value: line 1 column 1 (char 0)


7

## 4. Classification

What was perturbed, by keyword on title and summary. A study can be several things at once (a knockout in two strains). The matched text is kept so each call can be checked.

In [7]:
cat = pd.concat([geo, sra_only, pride, meta], ignore_index=True).drop_duplicates('accession')
TEXT = (cat['title'] + ' ' + cat['summary']).str.replace(r'\s+', ' ', regex=True)
RULES = {
    'gene perturbation': r'knock-?out|knock-?down|\bdelet(?:ion|ed)\b|\bnull mutant|\bmutant\b|\u0394\w|\bKO\b|auxin|\bm?AID\b|degron|conditional (?:knock|deplet)|deplet(?:ion|ed) of|CRISPR|tet-?off|ATc|overexpress',
    'drug or compound': r'\bdrugs?\b|\bcompounds?\b|inhibitors?\b|treated with (?!trypsin)|treatment with (?!trypsin)|pyrimethamine|sulfadiazine|atovaquone|artemisinin|clindamycin|endochin|buparvaquone|\bIC50\b|antiparasitic|anti-?toxoplasma',
    'strain comparison': r'\bstrains?\b.{0,80}?(?:type I\b|type II\b|type III|\bRH\b|ME49|PRU\b|VEG\b|GT1|CTG|CEP\b)|(?:type I\b|type II\b|type III).{0,60}?(?:type I\b|type II\b|type III)|\b(?:RH|GT1)\b.{0,60}?\b(?:ME49|PRU|VEG|CTG)\b',
    'stage or differentiation': r'bradyzoite|tachyzoite.{0,40}bradyzoite|differentiation|cyst|sporozoite|oocyst|merozoite|sexual|gametocyte|life cycle',
    'host response': r'host cell|infected (?:cells|mice|fibroblast|macrophage)|human foreskin|HFF|mouse brain|macrophage|dendritic',
}
for name, rx in RULES.items():
    m = TEXT.str.extract('(' + rx + ')', flags=re.I)[0]
    cat[name] = m.notna(); cat[name + ' (matched)'] = m.fillna('')
def readout(t):
    t = str(t).lower()
    for k, v in (('proteomics', 'proteome'), ('metabolomics', 'metabolome'), ('methylation', 'methylation'),
                 ('genome binding', 'chromatin / binding'), ('genome variation', 'genome'), ('non-coding', 'non-coding RNA'),
                 ('high throughput sequencing', 'RNA-seq / sequencing'), ('array', 'microarray')):
        if k in t: return v
    return 'other'
cat['readout'] = cat['type'].map(readout)
cat.groupby('readout')[list(RULES)].sum().assign(studies=cat.groupby('readout').size())

,gene perturbation,drug or compound,strain comparison,stage or differentiation,host response,studies
readout,,,,,,
RNA-seq / sequencing,84,32,23,83,74,269
chromatin / binding,19,4,1,32,6,56
genome,0,0,1,0,0,4
metabolome,0,1,0,3,1,7
methylation,0,0,1,0,5,7
microarray,27,7,16,24,47,110
non-coding RNA,0,2,1,2,4,12
other,12,2,0,8,3,18
proteome,52,20,9,47,62,209


## 5. The thioredoxin knockout and the strain comparisons asked for

Exact-word matches, listed whole so a person can read them.

In [8]:
trx = cat[TEXT.str.contains(r'thioredoxin|\bTRX\d?\b|\bTrxL', case=False, regex=True)]
trx[['source', 'accession', 'readout', 'samples', 'title']]

,source,accession,readout,samples,title
371,SRA,PRJNA1008424,RNA-seq / sequencing,None,RNA-Seq of Toxoplasma gondii Trx1 mutant
498,PRIDE,PXD063659,proteome,None,A Protein Disulfide Isomerase Coordinates Redox Homeostasis and ER Calcium Regulation for Optimal Lytic Cycle Progression in Toxoplasma gondii. SERCA-HA DVSF HA-IP LC-MS/MS
499,PRIDE,PXD063619,proteome,None,A Protein Disulfide Isomerase Coordinates Redox Homeostasis and ER Calcium Regulation for Optimal Lytic Cycle Progression in Toxoplasma gondii. streptavadin-IP LC-MS/MS
500,PRIDE,PXD063615,proteome,None,A Protein Disulfide Isomerase Coordinates Redox Homeostasis and ER Calcium Regulation for Optimal Lytic Cycle Progression in Toxoplasma gondii. DVSF anti-TgPDIA3 LC-MS/MS


In [9]:
strains = cat[cat['strain comparison'] & cat['parasite_measured']]
strains[['source', 'accession', 'readout', 'samples', 'strain comparison (matched)', 'title']].sort_values('samples', ascending=False)

,source,accession,readout,samples,strain comparison (matched),title
38,GEO,GSE26558,microarray,80,Type II X III and Type I,Expression Quantitative Trait Locus (eQTL) Mapping of Stage-specific Gene Expression in Progeny from a type I X III Genetic Cross of Toxoplasma gondii
33,GEO,GSE24905,genome,49,type I lineage and moderately virulent type II,Virulence differences in Toxoplasma mediated by amplification of polymorphic pseudokinases.
260,GEO,GSE218921,RNA-seq / sequencing,27,strain VEG,"In vitro analysis of pH responsiveness of T. gondii and H. hammondi sporozoites, and of T. gondii tachyzoites lacking either the BFD1 or the ROCY1 gene."
91,GEO,GSE60305,RNA-seq / sequencing,18,strains RH,"RNA-seq transcriptional profile of Toxoplasma gondii strains RH, PLK, CTG as tachyzoites and bradyzoites"
340,GEO,GSE294543,methylation,18,"RH (type I), ME49",Cross-lineage m5C methylome profiling reveals methylated divergence in Toxoplasma tachyzoites
119,GEO,GSE93531,microarray,16,type I RH diCRE (RHCre) and type II,Analysis of the role for the transcription factor TgAP2IV-4 (TGGT1_318470) in regulating gene expression in Toxoplasma development
235,GEO,GSE200962,RNA-seq / sequencing,16,RH and slower dividing developmentally competent ME49,RESTRICTION checkpoint controls bradyzoite development in Toxoplasma gondii
20,GEO,GSE16037,microarray,12,"GT1, II Me49","Types I GT1, II Me49, III CTGara strains - tachyzoite and Compound 1"
131,GEO,GSE103113,RNA-seq / sequencing,8,"strain (ToxoDB-24, ME49",Transcriptomic analysis by Next Generation Sequencing of mouse bone marrow derived macrophages (BMDMs) infected by Wild-Type and gra18 mutant strains of T. gondii.
207,GEO,GSE159515,RNA-seq / sequencing,6,"RH tachyzoites, as well as from ME49",Genome-wide mapping of transcription initiation in the asexual stages of Toxoplasma gondii


## 6. Gene perturbations and drugs measured on the parasite

The pool for loading: the parasite is measured (not only the host), and something was knocked out, knocked down or dosed. Samples is the series total; replicate structure is checked per study when it is loaded.

In [10]:
pool = cat[cat['parasite_measured'] & (cat['gene perturbation'] | cat['drug or compound'])]
print(len(pool), 'studies;', pool.groupby('readout').size().to_dict())
pool[['source', 'accession', 'readout', 'samples', 'gene perturbation (matched)', 'drug or compound (matched)', 'title']].sort_values(['readout', 'samples'], ascending=[True, False]).head(60)

197 studies; {'RNA-seq / sequencing': 86, 'chromatin / binding': 21, 'metabolome': 1, 'microarray': 15, 'other': 12, 'proteome': 62}


,source,accession,readout,samples,gene perturbation (matched),drug or compound (matched),title
273,GEO,GSE229505,RNA-seq / sequencing,36,CRISPR,,High-throughput identification of Toxoplasma gondii effector proteins that target host cell transcription
266,GEO,GSE223819,RNA-seq / sequencing,28,knockdown,,"Effect of depleting BFD1-regulated gene-expression factors (TGME49_253790, TGME49_208020, TGME49_224630, TGME49_306620, TGME49_311100) on the Toxoplasma transcriptome during alkaline stress"
222,GEO,GSE178355,RNA-seq / sequencing,27,Depletion of,,Depletion of m6A writers and readers impair 3'-end formation in Toxoplasma
267,GEO,GSE223869,RNA-seq / sequencing,24,deletion,,Effect of BFD2 (TGME49_311100) deletion on gene expression in Toxoplasma gondii
268,GEO,GSE223877,RNA-seq / sequencing,24,knockout,treatment with,Effect of BFD2 (TGME49_311100) knockout on T. gondii gene expression during BFD1 induction
283,GEO,GSE243207,RNA-seq / sequencing,24,Depletion of,,Depletion of mRNA cap-binding protein eIF4E1 drives bradyzoite formation in Toxoplasma gondii
284,GEO,GSE245775,RNA-seq / sequencing,24,overexpress,,Translation initiation factor eIF1.2 orchestrates Toxoplasma gondii differentiation by regulating stage-specific gene expression
219,GEO,GSE168465,RNA-seq / sequencing,21,,drug,Primary brain cell infection by Toxoplasma gondii reveals the dynamics of spontaneous bradyzoite differentiation and the extend of modification of neuron biology after infection
182,GEO,GSE136123,RNA-seq / sequencing,18,conditional deplet,,A MORC-driven transcriptional switch controls Toxoplasma developmental trajectories and sexual commitment
354,GEO,GSE313582,RNA-seq / sequencing,18,auxin,,Divergent SAGA Complexes Shape the Toxoplasma Transcriptome across Host-Responsive and Core Gene Networks [RNAseq]


In [11]:
cat.to_csv(os.path.join(OUT, 'catalogue.csv'), index=False)
pd.Series(QUERIES).to_csv(os.path.join(OUT, 'queries.txt'), index=False, header=False)
len(cat), len(QUERIES)

(692, 17)